In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.bias import Balance
from dataeval.config import set_max_processes
from dataeval.core import image_stats
from dataeval.data import Limit, Shuffle, View
from dataeval.flags import ImageStats

# Read and measure four frames at a time rather than one. See the "Measure your own factors" section.
set_max_processes(4)

In [3]:
dataset = View(SeaDrone(root="./data", image_set="val", download=True), [Shuffle(seed=0), Limit(200)])
metadata = Metadata(dataset)

print("levels      :", metadata.levels)
print("level counts:", metadata.level_counts)

levels      : ('unit', 'instance')
level counts: {'unit': 200, 'instance': 1305}


/tmp/ipykernel_6064/2048650437.py:4: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  print("levels      :", metadata.levels)


In [4]:
EXPECTED_COUNTS = {"unit": 200, "instance": 1305}

if dict(metadata.level_counts) != EXPECTED_COUNTS:
    raise AssertionError(
        f"This tutorial's prose was written against {EXPECTED_COUNTS} (maite-datasets 0.0.21), "
        f"but your dataset yields {dict(metadata.level_counts)}. Every idea below still holds — "
        "levels, weighting, aggregation — but the specific figures quoted will not match your output."
    )

In [5]:
print("one row per image:")
print(metadata.rows_at("unit").select("item_index", "altitude", "speed").head(3))

print("\none row per detection:")
print(metadata.rows_at("instance").select("item_index", "class_label", "object_size").head(3))

one row per image:
shape: (3, 3)
┌────────────┬───────────┬───────┐
│ item_index ┆ altitude  ┆ speed │
│ ---        ┆ ---       ┆ ---   │
│ i64        ┆ f64       ┆ f64   │
╞════════════╪═══════════╪═══════╡
│ 0          ┆ 17.099166 ┆ 0.0   │
│ 1          ┆ 9.099556  ┆ 0.0   │
│ 2          ┆ -1.0      ┆ -1.0  │
└────────────┴───────────┴───────┘

one row per detection:
shape: (3, 3)
┌────────────┬─────────────┬─────────────┐
│ item_index ┆ class_label ┆ object_size │
│ ---        ┆ ---         ┆ ---         │
│ i64        ┆ i64         ┆ i64         │
╞════════════╪═════════════╪═════════════╡
│ 0          ┆ 2           ┆ 782         │
│ 0          ┆ 2           ┆ 2916        │
│ 0          ┆ 2           ┆ 1392        │
└────────────┴─────────────┴─────────────┘


/tmp/ipykernel_6064/2930514107.py:2: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `object_id`, `object_size`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  print(metadata.rows_at("unit").select("item_index", "altitude", "speed").head(3))


In [6]:
valid = pl.col("altitude") >= 0
per_image = metadata.rows_at("unit").filter(valid)["altitude"]
per_detection = metadata.rows_at("instance").filter(valid)["altitude"]

print(f"per image     n={len(per_image):5d}  mean={per_image.mean():.1f} m  std={per_image.std():.1f}")
print(f"per detection n={len(per_detection):5d}  mean={per_detection.mean():.1f} m  std={per_detection.std():.1f}")

per image     n=  142  mean=41.8 m  std=29.8
per detection n=  922  mean=48.1 m  std=30.8


In [7]:
info = metadata.factor_info["altitude"]
print(f"altitude is defined at the {info.level!r} level, binned: {info.is_binned}")

altitude is defined at the 'unit' level, binned: True


In [8]:
# TEST ASSERTION CELL ###
from dataclasses import dataclass

import numpy as np

from dataeval.protocols import DatasetMetadata, DatumMetadata


def edges(values):
    cut = Metadata.from_factors(
        {"altitude": values}, class_labels=np.zeros(len(values), dtype=int), auto_bin_method="uniform_count"
    )
    return np.array(cut.encoding("altitude").edges)


# the edges come from the 200 image values and not from the 1305 detection rows
image_values = metadata.rows_at("unit")["altitude"].to_numpy()
detection_values = metadata.rows_at("instance")["altitude"].to_numpy()
own = np.array(Metadata(dataset, auto_bin_method="uniform_count").encoding("altitude").edges)
assert np.array_equal(own, edges(image_values))
assert not np.array_equal(own, edges(detection_values))

# an image's bin is the same number on its own row and on each of its detections
by_item = dict(zip(metadata.rows_at("unit")["item_index"], metadata.rows_at("unit")["altitude↕"], strict=True))
detections = metadata.rows_at("instance")
assert all(by_item[item] == code for item, code in zip(detections["item_index"], detections["altitude↕"], strict=True))


# an image with no detections still gets a bin
@dataclass
class BoxTarget:
    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray


class Sparse:
    counts = (2, 1, 2, 0)
    metadata = DatasetMetadata(id="sparse", index2label={0: "a", 1: "b"})

    def __len__(self):
        return len(self.counts)

    def __getitem__(self, index):
        count = self.counts[index]
        target = BoxTarget(np.tile([[0.0, 0.0, 10.0, 10.0]], (count, 1)), np.arange(count) % 2, np.ones(count))
        return np.zeros((3, 8, 8), np.uint8), target, DatumMetadata(id=index, **{"altitude_m": float(100 * index)})


sparse = Metadata(Sparse(), auto_bin_method="uniform_count")
info = sparse.factor_info["altitude_m"]
column = "altitude_m↕" if info.is_binned else "altitude_m#" if info.is_digitized else "altitude_m"
assert sparse.level_counts == {"unit": 4, "instance": 5}
assert sparse.rows_at("unit")[column].null_count() == 0

/tmp/ipykernel_6064/2491927326.py:19: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  own = np.array(Metadata(dataset, auto_bin_method="uniform_count").encoding("altitude").edges)
/tmp/ipykernel_6064/2491927326.py:19: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `object_id`, `object_size`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_count') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  own = np.array(Metadata(dataset, auto_bin_method="uniform_count").encoding("altitude"

In [9]:
print("factor_data at instance (default):", metadata.factor_data.shape)
print("factor_data at unit               :", metadata.at("unit").factor_data.shape)

factor_data at instance (default): (1305, 15)
factor_data at unit               : (200, 13)


In [10]:
try:
    labels_per_image = metadata.at("unit").class_labels
except ValueError as error:
    print("as expected:", str(error).split(".")[0])

as expected: class_labels is defined at the 'instance' level, but this metadata is viewed at 'unit', which has no label per row


In [11]:
own_only = Metadata(dataset, inherited=False)

print("at instance, inherited=True :", len(metadata.factor_names), "factors")
print("at instance, inherited=False:", sorted(own_only.factor_names))

at instance, inherited=True : 15 factors


at instance, inherited=False: ['object_id', 'object_size']


In [12]:
stats = image_stats(
    dataset,
    stats=ImageStats.VISUAL_SHARPNESS | ImageStats.VISUAL_BRIGHTNESS,
    per_image=True,
    per_target=True,
    per_background=True,
    normalize_pixel_values=False,
)

print("measured:", sorted(stats["stats"]))

measured: ['background_brightness', 'background_fraction', 'background_sharpness', 'brightness', 'sharpness']


In [13]:
extrinsic = set(metadata.factor_names)
metadata.add_factors(stats)

print("new factors:", sorted(set(metadata.factor_names) - extrinsic))
print("dropped    :", metadata.dropped_factors)

new factors: ['instance_brightness', 'instance_sharpness', 'unit_background_brightness', 'unit_background_fraction', 'unit_background_sharpness', 'unit_brightness', 'unit_sharpness']
dropped    : {'latitude': ['mixed_types'], 'longitude': ['mixed_types'], 'date_time': ['cardinality_over_budget'], 'instance_background_brightness': ['no_values_at_level'], 'instance_background_sharpness': ['no_values_at_level'], 'instance_background_fraction': ['no_values_at_level']}


In [14]:
print(
    metadata
    .rows_at("unit")
    .select("item_index", "unit_sharpness", "unit_background_sharpness", "instance_sharpness")
    .head(3)
)

print("\nand from the detection rows, where the image value propagates down:")
print(metadata.rows_at("instance").select("item_index", "unit_sharpness", "instance_sharpness").head(3))

shape: (3, 4)
┌────────────┬────────────────┬───────────────────────────┬────────────────────┐
│ item_index ┆ unit_sharpness ┆ unit_background_sharpness ┆ instance_sharpness │
│ ---        ┆ ---            ┆ ---                       ┆ ---                │
│ i64        ┆ f32            ┆ f32                       ┆ f32                │
╞════════════╪════════════════╪═══════════════════════════╪════════════════════╡
│ 0          ┆ 10.87645       ┆ 10.781616                 ┆ null               │
│ 1          ┆ 22.498152      ┆ 19.834988                 ┆ null               │
│ 2          ┆ 7.374899       ┆ 7.114103                  ┆ null               │
└────────────┴────────────────┴───────────────────────────┴────────────────────┘

and from the detection rows, where the image value propagates down:
shape: (3, 3)
┌────────────┬────────────────┬────────────────────┐
│ item_index ┆ unit_sharpness ┆ instance_sharpness │
│ ---        ┆ ---            ┆ ---                │
│ i64        ┆ 

/tmp/ipykernel_6064/1177355039.py:3: UserWarning: `instance_brightness`, `instance_sharpness`, `unit_background_brightness`, `unit_background_fraction`, `unit_background_sharpness`, `unit_brightness` and `unit_sharpness` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"instance_brightness": [...]} to control this.
  .rows_at("unit")


In [15]:
enriched = metadata.agg(
    "instance",
    "unit",
    pl.len().alias("n_objects"),
    pl.col("object_size").mean().alias("mean_object_size"),
    pl.col("instance_sharpness").mean().alias("mean_object_sharpness"),
)

new_factors = sorted(set(enriched.at("unit").factor_names) - set(metadata.at("unit").factor_names))
print("new factors at the image level:", new_factors)
print(enriched.rows_at("unit").select("item_index", "altitude", "n_objects", "mean_object_size").head(5))

new factors at the image level: ['mean_object_sharpness', 'mean_object_size', 'n_objects']
shape: (5, 4)
┌────────────┬───────────┬───────────┬──────────────────┐
│ item_index ┆ altitude  ┆ n_objects ┆ mean_object_size │
│ ---        ┆ ---       ┆ ---       ┆ ---              │
│ i64        ┆ f64       ┆ u32       ┆ f64              │
╞════════════╪═══════════╪═══════════╪══════════════════╡
│ 0          ┆ 17.099166 ┆ 5         ┆ 1126.0           │
│ 1          ┆ 9.099556  ┆ 1         ┆ 226032.0         │
│ 2          ┆ -1.0      ┆ 5         ┆ 12361.0          │
│ 3          ┆ 29.798547 ┆ 3         ┆ 9713.666667      │
│ 4          ┆ 17.099166 ┆ 5         ┆ 1184.0           │
└────────────┴───────────┴───────────┴──────────────────┘


/tmp/ipykernel_6064/2239674715.py:9: UserWarning: `mean_object_sharpness` and `mean_object_size` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"mean_object_sharpness": [...]} to control this.
  new_factors = sorted(set(enriched.at("unit").factor_names) - set(metadata.at("unit").factor_names))


In [16]:
crowding = enriched.rows_at("unit").filter(valid)
print("altitude vs objects-per-image, correlation:")
print(round(float(crowding.select(pl.corr("altitude", "n_objects")).item()), 3))

altitude vs objects-per-image, correlation:
0.333


In [17]:
print(
    enriched
    .rows_at("unit")
    .select("item_index", "n_objects", "unit_sharpness", "unit_background_sharpness", "mean_object_sharpness")
    .head(5)
)

shape: (5, 5)
┌────────────┬───────────┬────────────────┬───────────────────────────┬───────────────────────┐
│ item_index ┆ n_objects ┆ unit_sharpness ┆ unit_background_sharpness ┆ mean_object_sharpness │
│ ---        ┆ ---       ┆ ---            ┆ ---                       ┆ ---                   │
│ i64        ┆ u32       ┆ f32            ┆ f32                       ┆ f32                   │
╞════════════╪═══════════╪════════════════╪═══════════════════════════╪═══════════════════════╡
│ 0          ┆ 5         ┆ 10.87645       ┆ 10.781616                 ┆ 50.692177             │
│ 1          ┆ 1         ┆ 22.498152      ┆ 19.834988                 ┆ 61.729645             │
│ 2          ┆ 5         ┆ 7.374899       ┆ 7.114103                  ┆ 33.115841             │
│ 3          ┆ 3         ┆ 30.256311      ┆ 30.112347                 ┆ 55.972309             │
│ 4          ┆ 5         ┆ 11.442901      ┆ 11.321176                 ┆ 55.266666             │
└────────────┴───────────┴

In [18]:
per_object = Balance().evaluate(metadata)
ranked = per_object.balance.sort("mi_value", descending=True)
print(ranked.head(5))

print("\nobject_size, for comparison:")
print(ranked.filter(pl.col("factor_name") == "object_size"))

shape: (5, 2)
┌──────────────┬──────────┐
│ factor_name  ┆ mi_value │
│ ---          ┆ ---      │
│ cat          ┆ f64      │
╞══════════════╪══════════╡
│ class_label  ┆ 1.0      │
│ storage      ┆ 0.317702 │
│ object_id    ┆ 0.221884 │
│ gimbal_pitch ┆ 0.21591  │
│ altitude     ┆ 0.203709 │
└──────────────┴──────────┘

object_size, for comparison:
shape: (1, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ object_size ┆ 0.033163 │
└─────────────┴──────────┘


In [19]:
per_image_balance = Balance(label="n_objects").evaluate(enriched.at("unit"))
print(per_image_balance.balance.sort("mi_value", descending=True).head(5))

shape: (5, 2)
┌──────────────────────────┬──────────┐
│ factor_name              ┆ mi_value │
│ ---                      ┆ ---      │
│ cat                      ┆ f64      │
╞══════════════════════════╪══════════╡
│ n_objects                ┆ 1.0      │
│ storage                  ┆ 0.49952  │
│ mean_object_size         ┆ 0.265248 │
│ gimbal_pitch             ┆ 0.23923  │
│ unit_background_fraction ┆ 0.217276 │
└──────────────────────────┴──────────┘


In [20]:
swimmer_rows = metadata.where(pl.col("class_label") == 1, level="instance")
swimmer_images = metadata.having(pl.col("class_label") == 1, level="instance")

print("unfiltered:", metadata.level_counts)
print("where     :", swimmer_rows.level_counts)
print("having    :", swimmer_images.level_counts)

unfiltered: {'unit': 200, 'instance': 1305}
where     : {'unit': 200, 'instance': 907}
having    : {'unit': 178, 'instance': 1235}


In [21]:
high = metadata.where(pl.col("altitude") > 60, level="unit")
print("altitude > 60 m:", high.level_counts)

altitude > 60 m: {'unit': 22, 'instance': 192}
